In [1]:
from dotenv import load_dotenv
from pathlib import Path
import sys
import os

# Walk up until we find the project root (folder with the .env)
current_path = Path().resolve()
for parent in [current_path] + list(current_path.parents):
    if (parent / ".env").exists():
        load_dotenv(parent / ".env")
        project_root = os.getenv("PROJECT_ROOT")
        print(project_root)
        sys.path.append(project_root)     
        break


%load_ext autoreload
%autoreload 2

/Users/emmanuel/Documents/belugas/beluga-call-pipeline


In [2]:
import pandas as pd

import torch
from models.resnet import ResnetMultilabel
from models.mobilenet import MobileNetMultilabel
from models.quant_mobilenet import load_mobilenet_v3_quant

from training.cross_validation import run_cross_val, train_model


## Running the Optimization Experiments

This section covers the model optimization experiments:
- Switching from **ResNet18** to **MobileNet V3 Small**,
- Further **Truncating** the MobileNet architecture,
- Applying **8-bit Quantization-Aware Training (QAT)** on the MobileNet model.

Each experiment is executed with **cross-validation** as described in the paper.  
Results are written to a dedicated `results/` directory and subsequently examined in the `results_analysis` folder.



In [3]:
labels_df = pd.read_csv("../data/Verified_Dataset/labels/labels_merged.csv")
labels_df["ClipFilenamePt"] = labels_df["clip_filename"].str.replace(".wav", ".pt", regex=False)


label_columns = ["ECHO", "HFPC", "BBPC", "Whistle"]

data_dir = "../data"
processed_spects_dir = data_dir + "/Verified_Dataset/spectrograms/"

results_dir = "./results/new_dataset"

In [4]:
from training.cross_validation import create_test_fold_indices
labels_df = create_test_fold_indices(labels_df, 5)

In [7]:
labels_df["Site_Day"] = labels_df["Site"] + "_" + pd.to_datetime(labels_df["clip_start_time"]).dt.strftime("%Y-%m-%d")

In [12]:
labels_df[labels_df["Boat"]==1]["original_filename"].unique()

array([nan, '201359382.170724133002.wav', '201359382.210714075958.wav',
       '5725.200726185951.wav'], dtype=object)

In [11]:
pd.set_option('display.max_columns', None)
labels_df

,clip_filename,ECHO,BBPC,HFPC,Whistle,Call,Boat,labeling_effort,DETAIL,Begin Time (s),End Time (s),GROUNDTRUTH,Notes,original_filename,labeled_snippet_filename,snippet_start_time,snippet_start_s,snippet_end_s,Site,labeled_snippet_dir,HydrophoneModel,HydrophoneSensitivity,clip_start_time,clip_end_time,SnippetFilename,start_s,end_s,ClipFilenamePt,test_fold_idx,Site_Day
0,BSM_20170724_09480100.wav,0,0,0,0,0,NaN,old_abs_from_3s_snippets,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017-07-24 09:48:00.000,NaN,NaN,BSM,NaN,201359382,-172.7,2017-07-24 09:48:01.000,2017-07-24 09:48:02.000,BSM_20170724_094800.wav,0.0,1.0,BSM_20170724_09480100.pt,4,BSM_2017-07-24
1,BSM_20170724_09480200.wav,0,0,0,0,0,NaN,old_abs_from_3s_snippets,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017-07-24 09:48:00.000,NaN,NaN,BSM,NaN,201359382,-172.7,2017-07-24 09:48:02.000,2017-07-24 09:48:03.000,BSM_20170724_094800.wav,1.0,2.0,BSM_20170724_09480200.pt,3,BSM_2017-07-24
2,BSM_20170724_09480300.wav,0,0,0,0,0,NaN,old_abs_from_3s_snippets,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017-07-24 09:48:00.000,NaN,NaN,BSM,NaN,201359382,-172.7,2017-07-24 09:48:03.000,2017-07-24 09:48:04.000,BSM_20170724_094800.wav,2.0,3.0,BSM_20170724_09480300.pt,0,BSM_2017-07-24
3,BSM_20170724_10580100.wav,0,0,0,0,0,NaN,old_abs_from_3s_snippets,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017-07-24 10:58:00.000,NaN,NaN,BSM,NaN,201359382,-172.7,2017-07-24 10:58:01.000,2017-07-24 10:58:02.000,BSM_20170724_105800.wav,0.0,1.0,BSM_20170724_10580100.pt,2,BSM_2017-07-24
4,BSM_20170724_10580200.wav,0,0,0,0,0,NaN,old_abs_from_3s_snippets,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017-07-24 10:58:00.000,NaN,NaN,BSM,NaN,201359382,-172.7,2017-07-24 10:58:02.000,2017-07-24 10:58:03.000,BSM_20170724_105800.wav,1.0,2.0,BSM_20170724_10580200.pt,1,BSM_2017-07-24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11603,RDL_20200906_14555938.wav,0,0,0,0,0,NaN,old_abs_from_3s_snippets,NaN,NaN,NaN,NaN,NaN,5675.200906144958.wav,NaN,2020-09-06 14:55:57.380000,NaN,NaN,RDL,NaN,5675,-176.5,2020-09-06 14:55:59.380000,2020-09-06 14:56:00.380000,RDL_20200906_14555738.wav,1.0,2.0,RDL_20200906_14555938.pt,4,RDL_2020-09-06
11604,RDL_20200906_14560038.wav,0,0,0,0,0,NaN,old_abs_from_3s_snippets,NaN,NaN,NaN,NaN,NaN,5675.200906144958.wav,NaN,2020-09-06 14:55:57.380000,NaN,NaN,RDL,NaN,5675,-176.5,2020-09-06 14:56:00.380000,2020-09-06 14:56:01.380000,RDL_20200906_14555738.wav,2.0,3.0,RDL_20200906_14560038.pt,0,RDL_2020-09-06
11605,RDL_20200906_14573769.wav,0,0,0,0,0,NaN,old_abs_from_3s_snippets,NaN,NaN,NaN,NaN,NaN,5675.200906144958.wav,NaN,2020-09-06 14:57:36.690000,NaN,NaN,RDL,NaN,5675,-176.5,2020-09-06 14:57:37.690000,2020-09-06 14:57:38.690000,RDL_20200906_14573669.wav,0.0,1.0,RDL_20200906_14573769.pt,0,RDL_2020-09-06
11606,RDL_20200906_14573869.wav,0,0,0,0,0,NaN,old_abs_from_3s_snippets,NaN,NaN,NaN,NaN,NaN,5675.200906144958.wav,NaN,2020-09-06 14:57:36.690000,NaN,NaN,RDL,NaN,5675,-176.5,2020-09-06 14:57:38.690000,2020-09-06 14:57:39.690000,RDL_20200906_14573669.wav,1.0,2.0,RDL_20200906_14573869.pt,4,RDL_2020-09-06


In [5]:
labels_df["test_fold_idx"].value_counts()

test_fold_idx
0    2322
2    2322
1    2322
4    2321
3    2321
Name: count, dtype: int64